# Intro 06 — Continuous Random Variables

Practice notebook for the **"Continuous Random Variables"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Probability Density Function (PDF) and CDF

A **continuous** random variable \(X\) has a **probability density function** (PDF) \(f_X(x)\) such that,
for any interval \((a,b)\),
$$P(a < X < b) = \int_{a}^{b} f_X(x)\,dx.$$

The PDF must satisfy
$$f_X(x) \geq 0 \quad\text{for all } x, \qquad \int_{-\infty}^{\infty} f_X(x)\,dx = 1.$$

For continuous \(X\), the probability at a single point is **zero**:
$$P(X = x_0) = 0,$$
so only intervals have positive probability.

The **cumulative distribution function** (CDF) packages up all interval probabilities:
$$F_X(x) = P(X \leq x) = \int_{-\infty}^{x} f_X(t)\,dt.$$
Then \(P(a < X < b) = F_X(b) - F_X(a)\).

Below we use SciPy's `norm` distribution to plot a PDF and its CDF together, and we confirm
that the area under the PDF over a wider and wider window approaches 1 (numerical normalization).


In [ ]:
# Plot the PDF and CDF of a standard normal side by side, and verify normalization
x = np.linspace(-5, 5, 1000)
pdf = stats.norm.pdf(x, loc=0, scale=1)
cdf = stats.norm.cdf(x, loc=0, scale=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x, pdf, lw=2, label="f_X(x)")
axes[0].fill_between(x, pdf, alpha=0.15)
axes[0].set_title("Standard normal PDF")
axes[0].set_xlabel("x"); axes[0].set_ylabel("density")
axes[0].legend()

axes[1].plot(x, cdf, lw=2, color="tab:green", label="F_X(x)")
axes[1].set_title("Standard normal CDF")
axes[1].set_xlabel("x"); axes[1].set_ylabel("P(X <= x)")
axes[1].legend()
plt.tight_layout(); plt.show()

# Numerical normalization: integral of PDF over [-A, A] -> 1 as A grows
for A in [1, 2, 3, 5, 10]:
    area, _ = stats.norm.expect(lambda t: 1.0, loc=0, scale=1,
                               lb=-A, ub=A)
    print(f"  integral over [{-A:>3}, {A:>3}] = {area:.6f}")

# P(a < X < b) = F(b) - F(a) for a standard normal
a, b = -1.96, 1.96
p_interval = stats.norm.cdf(b) - stats.norm.cdf(a)
print(f"\nP({a} < X < {b}) = F({b}) - F({a}) = {p_interval:.4f}  (about 0.95)")


---
## Part 2 — Uniform Distribution

\(X\) is **uniform** on \([a,b]\), written \(X\sim\text{Uniform}(a,b)\), if
$$f_X(x) = \begin{cases}\dfrac{1}{b-a}, & a\leq x\leq b,\\ 0, & \text{otherwise}.\end{cases}$$
All points in \([a,b]\) are equally likely in density. The mean and variance are
$$E[X] = \frac{a+b}{2},\qquad \operatorname{Var}(X) = \frac{(b-a)^2}{12}.$$

**Example (from the PDF).** If \(X\sim\text{Uniform}(0,1)\), then
$$P(0.2 < X < 0.5) = \int_{0.2}^{0.5} 1\,dx = 0.5 - 0.2 = 0.3,$$
i.e. the chance of landing in a subinterval is just its **length**.

We plot the PDF/CDF of \(\text{Uniform}(0,1)\), confirm the example exactly, and compare a
large simulated sample's mean/variance to the theoretical values.


In [ ]:
# Uniform(a, b) = Uniform(0, 1)
a, b = 0.0, 1.0
U = stats.uniform(loc=a, scale=b - a)

x = np.linspace(-0.2, 1.2, 600)
pdf = U.pdf(x)
cdf = U.cdf(x)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, pdf, lw=2)
axes[0].set_title("Uniform(0,1) PDF  (height 1/(b-a) = 1)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("density"); axes[0].set_ylim(-0.1, 1.4)
axes[1].plot(x, cdf, lw=2, color="tab:green")
axes[1].set_title("Uniform(0,1) CDF")
axes[1].set_xlabel("x"); axes[1].set_ylabel("F_X(x)")
plt.tight_layout(); plt.show()

# Exact interval probability from the PDF/CDF
p_exact = U.cdf(0.5) - U.cdf(0.2)
print(f"P(0.2 < X < 0.5) = {p_exact:.4f}  (theory = 0.3)")

# Simulation: compare sample mean/variance to theory
rng = np.random.default_rng(42)
samples = rng.uniform(low=a, high=b, size=200_000)
mean_theory = (a + b) / 2
var_theory = (b - a) ** 2 / 12
print(f"sample mean   = {samples.mean():.5f}  (theory = {mean_theory})")
print(f"sample var    = {samples.var(ddof=1):.6f}  (theory = {var_theory:.6f})")


---
## Part 3 — Normal (Gaussian) Distribution

\(X\) is **normal** with mean \(\mu\) and variance \(\sigma^2\), written
\(X\sim N(\mu,\sigma^2)\), if
$$f_X(x) = \frac{1}{\sqrt{2\pi\sigma^2}}\,\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).$$
The bell curve is centered at \(\mu\); \(\sigma\) controls the spread. Key facts:
$$E[X] = \mu,\qquad \operatorname{Var}(X) = \sigma^2.$$

**Standardization.** If \(X\sim N(\mu,\sigma^2)\), then
$$Z = \frac{X - \mu}{\sigma} \sim N(0,1)$$
is the **standard normal**. Probabilities for any normal can be computed via \(Z\).

**The 68-95-99.7 rule.** For any normal,
$$P(\mu-\sigma < X < \mu+\sigma) \approx 0.6827,\quad
P(\mu-2\sigma < X < \mu+2\sigma) \approx 0.9545,\quad
P(\mu-3\sigma < X < \mu+3\sigma) \approx 0.9973.$$

We plot \(N(2, 3^2)\), standardize a sample, and verify both the mean/variance and the
68-95-99.7 rule against simulation.


In [ ]:
# Normal(mu, sigma^2) = Normal(2, 3^2)
mu, sigma = 2.0, 3.0
Ndist = stats.norm(loc=mu, scale=sigma)

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
pdf = Ndist.pdf(x)

fig, ax = plt.subplots()
ax.plot(x, pdf, lw=2, label=f"N({mu}, {sigma}^2) PDF")
# Shade the +/-1sigma, +/-2sigma, +/-3sigma regions
for k, color in zip([1, 2, 3], ["tab:orange", "tab:green", "tab:red"]):
    lo, hi = mu - k*sigma, mu + k*sigma
    m = (x >= lo) & (x <= hi)
    ax.fill_between(x[m], pdf[m], alpha=0.18, color=color,
                   label=f"$\\mu \\pm {k}\\sigma$")
ax.set_xlabel("x"); ax.set_ylabel("density")
ax.set_title("Normal(2, 9) and the 68-95-99.7 regions")
ax.legend()
plt.show()

# Verify the 68-95-99.7 rule exactly and via simulation
rng = np.random.default_rng(7)
S = rng.normal(loc=mu, scale=sigma, size=500_000)
for k in [1, 2, 3]:
    p_exact = Ndist.cdf(mu + k*sigma) - Ndist.cdf(mu - k*sigma)
    p_sim = np.mean((S > mu - k*sigma) & (S < mu + k*sigma))
    print(f"  k={k}:  theory = {p_exact:.4f},  simulated = {p_sim:.4f}")

print(f"\nsample mean   = {S.mean():.4f}  (theory mu = {mu})")
print(f"sample var     = {S.var(ddof=1):.4f}  (theory sigma^2 = {sigma**2:.4f})")

# Standardization: Z = (X - mu) / sigma should be N(0, 1)
Z = (S - mu) / sigma
print(f"\nStandardized sample: mean = {Z.mean():.4f}, std = {Z.std(ddof=1):.4f}  (target 0, 1)")
print(f"P(-1 < Z < 1) from standardized sample = {np.mean((Z > -1) & (Z < 1)):.4f}")


---
## Part 4 — Exponential Distribution & Memorylessness

The **exponential** distribution with rate \(\lambda>0\) has PDF
$$f_X(x) = \begin{cases}\lambda e^{-\lambda x}, & x\geq 0,\\ 0, & x<0.\end{cases}$$
It models waiting times between events in a Poisson process. Key facts:
$$E[X] = \frac{1}{\lambda},\qquad \operatorname{Var}(X) = \frac{1}{\lambda^2}.$$

**Example (from the PDF).** If phone calls arrive at an average rate of 3 per hour, the
waiting time \(X\) between calls is approximately exponential with \(\lambda=3\), so
$$E[X] = \frac{1}{3}\text{ hour}\approx 20\text{ minutes}.$$

**Memorylessness.** For \(s,t\geq 0\),
$$P(X > s + t \mid X > s) = P(X > t).$$
The distribution of the *remaining* wait is the same as the original — the past tells you
nothing about the future. We verify this identity numerically and by simulation.


In [ ]:
# Exponential(lambda) — SciPy parameterizes by scale = 1/lambda
lam = 3.0
Edist = stats.expon(scale=1.0 / lam)   # rate lambda = 3 per hour

x = np.linspace(0, 3 / lam, 1000)
pdf = Edist.pdf(x)
cdf = Edist.cdf(x)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, pdf, lw=2)
axes[0].set_title("Exponential($\\lambda=3$) PDF")
axes[0].set_xlabel("x (hours)"); axes[0].set_ylabel("density")
axes[1].plot(x, cdf, lw=2, color="tab:green")
axes[1].set_title("Exponential($\\lambda=3$) CDF")
axes[1].set_xlabel("x (hours)"); axes[1].set_ylabel("F_X(x)")
plt.tight_layout(); plt.show()

# Mean / variance: theory vs simulation
rng = np.random.default_rng(11)
S = rng.exponential(scale=1.0 / lam, size=300_000)
print(f"E[X]:  theory = {1/lam:.5f} h (~{60/lam:.1f} min),  simulated = {S.mean():.5f}")
print(f"Var[X]: theory = {1/lam**2:.6f},            simulated = {S.var(ddof=1):.6f}")

# Memorylessness: P(X > s + t | X > s) =? P(X > t)
s, t = 0.2, 0.3
p_surv_st = Edist.sf(s + t)            # P(X > s + t)
p_surv_s  = Edist.sf(s)                # P(X > s)
p_cond_theory = p_surv_st / p_surv_s   # conditional
p_t_theory = Edist.sf(t)               # P(X > t)
print(f"\nMemorylessness (theory):")
print(f"  P(X > {s+t}) / P(X > {s}) = {p_cond_theory:.5f}")
print(f"  P(X > {t})              = {p_t_theory:.5f}")

# Simulation check of the same identity
p_sim_st = np.mean(S > s + t)
p_sim_s  = np.mean(S > s)
p_sim_cond = p_sim_st / p_sim_s
p_sim_t  = np.mean(S > t)
print(f"\nMemorylessness (simulation, n=300k):")
print(f"  P(X > {s+t} | X > {s}) = {p_sim_cond:.4f}")
print(f"  P(X > {t})             = {p_sim_t:.4f}")


---
## Your turn

Before opening the Problem Set below:

- Pick a continuous distribution's parameters (e.g. a different \(\mu,\sigma\) for the
  normal, or a different rate \(\lambda\) for the exponential) and **plot** its PDF and CDF
  side by side.
- Verify \(\int f_X(x)\,dx = 1\) numerically by evaluating the CDF at a wide interval.
- For a normal of your choice, check the 68-95-99.7 rule both **exactly** (via `norm.cdf`) and
  **by simulation**.
- For an exponential of your choice, verify memorylessness for *two* different \((s,t)\)
  pairs using both the survival function and simulated samples.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Let \(X\sim\text{Uniform}(0,1)\). Using the CDF, compute \(P(0.2 < X < 0.5)\) and \(P(X > 0.7)\). Verify both with a simulation of 100,000 samples.

2. Let \(X\sim N(0,1)\). Using `scipy.stats.norm`, find \(P(X < 1)\), \(P(-1 < X < 1)\), and the value \(z\) such that \(P(Z > z) = 0.025\).

3. If \(X\sim N(\mu,\sigma^2)\) with \(\mu=100\) and \(\sigma=15\) (an IQ-like scale), what is \(P(85 < X < 130)\)? Standardize and express the answer as a probability involving the standard normal CDF \(\Phi\).

4. Phone calls arrive at rate \(\lambda=3\) per hour. What is the expected waiting time (in minutes) and the standard deviation of the waiting time? Compute \(P(X > 30\text{ min})\).

5. Verify the memorylessness identity \(P(X > s + t \mid X > s) = P(X > t)\) for an exponential with \(\lambda=2\) using \(s=0.5\), \(t=0.5\). Give both the exact survival-function values and a simulated estimate.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(P(0.2<X<0.5)=F(0.5)-F(0.2)=0.5-0.2=0.3\). \(P(X>0.7)=1-F(0.7)=0.3\). A simulation with `rng = np.random.default_rng(42)` and 100,000 samples gives estimates within about 0.005 of both values.

**2.** `stats.norm.cdf(1)` \(\approx 0.8413\). \(P(-1<X<1)=\Phi(1)-\Phi(-1)\approx 0.6827\). The upper-tail 2.5% quantile is `stats.norm.isf(0.025)` \(\approx 1.960\), i.e. \(z\approx 1.96\).

**3.** Standardize: \(Z=(X-100)/15\). \(P(85<X<130)=P(-1<Z<2)=\Phi(2)-\Phi(-1)\approx 0.9772 - 0.1587 = 0.8186\). Equivalently, `stats.norm.cdf(130, 100, 15) - stats.norm.cdf(85, 100, 15) ≈ 0.8186`.

**4.** \(E[X]=1/\lambda=1/3\) hour \(=20\) minutes; \(\operatorname{SD}(X)=\sqrt{1/\lambda^2}=1/\lambda=20\) minutes. \(P(X>0.5\text{ h})=e^{-\lambda\cdot 0.5}=e^{-1.5}\approx 0.2231\). Use `stats.expon.sf(0.5, scale=1/3)`.

**5.** With \(\lambda=2\), \(s=t=0.5\): \(P(X>1)=e^{-2}\approx 0.1353\); \(P(X>0.5)=e^{-1}\approx 0.3679\); conditional \(=0.1353/0.3679\approx 0.3679=e^{-1}=P(X>0.5)\). A simulation with `rng = np.random.default_rng(0)` and 200,000 samples gives \(P(X>1\mid X>0.5)\approx 0.368\), matching \(P(X>0.5)\approx 0.368\) ✓.

</details>
